# Embedding Visualization with UMAP and t-SNE
This notebook mirrors dataset usage in `eda_vit.ipynb` and focuses on embedding extraction + projection visualization using `embedding_visualization.py`.

In [ ]:
from pathlib import Path

ROOT_DIR = Path(".")
DATASET_DIR = ROOT_DIR / "dataset/Poles2025"

import numpy as np
np.random.seed(42)

In [ ]:
import importlib

from anomaly_detection import (
    build_embedding_extractor,
    extract_embeddings,
    list_images_by_split,
)
import embedding_visualization as ev

importlib.reload(ev)

ProjectionConfig = ev.ProjectionConfig
project_from_workflow = ev.project_from_workflow
plot_projection_scatter = ev.plot_projection_scatter
plot_projection_with_thumbnails = ev.plot_projection_with_thumbnails
plot_projection_interactive = ev.plot_projection_interactive

def build_workflow_from_extractor(
    subset_dir: Path,
    split_to_images_dir: dict[str, str],
    embedding_type: str = "vit",
    device: str | None = None,
    batch_size: int = 64,
    num_workers: int = 1,
):
    paths_by_split = list_images_by_split(subset_dir, split_to_images_dir)

    print(
        " | ".join([
            f"{split}={len(paths)}"
            for split, paths in paths_by_split.items()
        ])
    )

    extractor = build_embedding_extractor(
        embedding_type=embedding_type,
        device=device,
    )
    print("device:", extractor.device)

    embeddings_by_split: dict[str, np.ndarray] = {}
    string_paths_by_split: dict[str, list[str]] = {}

    for split, paths in paths_by_split.items():
        X, P = extract_embeddings(
            paths,
            extractor=extractor,
            batch_size=batch_size,
            num_workers=num_workers,
        )
        embeddings_by_split[split] = X
        string_paths_by_split[split] = P
        print(f"{split}: {X.shape}")

    return {
        "device": extractor.device,
        "backbone": extractor.backbone,
        "tfm": extractor.tfm,
        "extractor": extractor,
        "embeddings_by_split": embeddings_by_split,
        "paths_by_split": string_paths_by_split,
    }

def describe_extractor(workflow: dict):
    extractor = workflow["extractor"]
    print(f"extractor={type(extractor).__name__} | device={extractor.device}")

## Road_poles_iPhone

In [ ]:
TARGET_DATASET = "Road_poles_iPhone"
SPLIT_TO_IMAGES_DIR = {
    "train": "images/Train/train",
    "valid": "images/Validation/val",
    "test": "images/Test/test",
}
SUBSET_DIR = DATASET_DIR / TARGET_DATASET

workflow = build_workflow_from_extractor(
    subset_dir=SUBSET_DIR,
    split_to_images_dir=SPLIT_TO_IMAGES_DIR,
    embedding_type="vit",
)

describe_extractor(workflow)
for split, X in workflow["embeddings_by_split"].items():
    print(f"{split}: {X.shape}")

In [ ]:
umap_df = project_from_workflow(
    workflow,
    config=ProjectionConfig(
        method="umap",
        random_state=42,
        umap_n_neighbors=25,
        umap_min_dist=0.05,
    ),
)

plot_projection_interactive(
    umap_df,
    split_order=["train", "valid", "test"],
    title="Road_poles_iPhone - UMAP (interactive)",
    hover_content="thumbnail",
)

In [ ]:
tsne_df = project_from_workflow(
    workflow,
    config=ProjectionConfig(
        method="tsne",
        random_state=42,
        tsne_perplexity=100,
        tsne_n_iter=1000,
    ),
)

plot_projection_interactive(
    tsne_df,
    split_order=["train", "valid", "test"],
    title="Road_poles_iPhone - t-SNE (interactive)",
    hover_content="thumbnail",
)

## roadpoles_v1

In [ ]:
TARGET_DATASET = "roadpoles_v1"
SPLIT_TO_IMAGES_DIR = {
    "train": "train/images",
    "valid": "valid/images",
    "test": "test/images",
}
SUBSET_DIR = DATASET_DIR / TARGET_DATASET

workflow = build_workflow_from_extractor(
    subset_dir=SUBSET_DIR,
    split_to_images_dir=SPLIT_TO_IMAGES_DIR,
    embedding_type="vit",
)

describe_extractor(workflow)

umap_df = project_from_workflow(
    workflow,
    config=ProjectionConfig(method="umap", random_state=42),
)

plot_projection_interactive(
    umap_df,
    split_order=["train", "valid", "test"],
    title="roadpoles_v1 - UMAP (interactive)",
    hover_content="thumbnail",
)

In [ ]:
tsne_df = project_from_workflow(
    workflow,
    config=ProjectionConfig(method="tsne", random_state=42),
)

plot_projection_interactive(
    tsne_df,
    split_order=["train", "valid", "test"],
    title="roadpoles_v1 - t-SNE (interactive)",
    hover_content="thumbnail",
)

## RoadPoles-MSJ (merged test split)

In [ ]:
import shutil

SOURCE_SUBSET_DIR = DATASET_DIR / "RoadPoles-MSJ"
MERGED_SUBSET_DIR = DATASET_DIR / "RoadPoles-MSJ-merged"
MERGED_TEST_DIR = MERGED_SUBSET_DIR / "test"
MERGED_TEST_DIR.mkdir(parents=True, exist_ok=True)

copied = 0
for subdir in sorted(SOURCE_SUBSET_DIR.glob("*")):
    if not subdir.is_dir():
        continue
    for img_path in sorted(subdir.glob("*.jpg")):
        target_name = f"{subdir.name}_{img_path.name}"
        target_path = MERGED_TEST_DIR / target_name
        if not target_path.exists():
            shutil.copy2(img_path, target_path)
            copied += 1

print(f"Merged images copied: {copied}")
print(f"Merged test directory: {MERGED_TEST_DIR}")

In [ ]:
TARGET_DATASET = "RoadPoles-MSJ-merged"
SPLIT_TO_IMAGES_DIR = {"test": "test"}
SUBSET_DIR = DATASET_DIR / TARGET_DATASET

workflow = build_workflow_from_extractor(
    subset_dir=SUBSET_DIR,
    split_to_images_dir=SPLIT_TO_IMAGES_DIR,
    embedding_type="vit",
)

describe_extractor(workflow)

In [ ]:
umap_df = project_from_workflow(
    workflow,
    config=ProjectionConfig(method="umap", random_state=42),
)

plot_projection_interactive(
    umap_df,
    split_order=["test"],
    title="RoadPoles-MSJ-merged - UMAP (interactive)",
    hover_content="thumbnail",
    rotate_270=True,
)

In [ ]:
tsne_df = project_from_workflow(
    workflow,
    config=ProjectionConfig(method="tsne", random_state=42),
)

plot_projection_interactive(
    tsne_df,
    split_order=["train", "valid", "test"],
    title="RoadPoles-MSJ-merged - t-SNE (interactive)",
    hover_content="thumbnail",
    rotate_270=True,
)